In [145]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import sqlite3

In [146]:
pd.set_option("display.max_columns", 50)
customer_churn = pd.read_csv("../data/Telco_Customer_Churn.csv")
customer_churn.head(10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [147]:
customer_churn.shape

(7043, 21)

In [148]:
customer_churn.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

In [149]:
customer_churn.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [150]:
customer_churn.select_dtypes(include=['number'])

,SeniorCitizen,tenure,MonthlyCharges
0,0,1,29.85
1,0,34,56.95
2,0,2,53.85
3,0,45,42.30
4,0,2,70.70
...,...,...,...
7038,0,24,84.80
7039,0,72,103.20
7040,0,11,29.60
7041,1,4,74.40


In [151]:
customer_churn['TotalCharges'] = pd.to_numeric(customer_churn['TotalCharges'],errors='coerce')

In [152]:
customer_churn.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [153]:
customer_churn['SeniorCitizen'] = customer_churn['SeniorCitizen'].astype('object')

In [154]:
customer_churn.dtypes

customerID           object
gender               object
SeniorCitizen        object
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

In [155]:
boolean_columns = [col for col in customer_churn.columns if customer_churn[col].nunique()==2]
boolean_columns

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'PhoneService',
 'PaperlessBilling',
 'Churn']

In [156]:
# Map each column based on its actual values
for col in boolean_columns:
    unique_vals = set(customer_churn[col].dropna().unique())
    if unique_vals.issubset({'Yes', 'No'}) or unique_vals.issubset({'Female','Male'}):  # Only map if values are Yes/No
        customer_churn[col] = customer_churn[col].map({'Yes': 1, 'No': 0,'Female':0,'Male':1})
    # Columns with 0/1 values are left unchanged

customer_churn[boolean_columns].head()

,gender,SeniorCitizen,Partner,Dependents,PhoneService,PaperlessBilling,Churn
0,0,0,1,0,0,1,0
1,1,0,0,0,1,0,0
2,1,0,0,0,1,1,1
3,1,0,0,0,0,0,0
4,0,0,0,0,1,1,1


In [157]:
customer_churn[boolean_columns].values

array([[0, 0, 1, ..., 0, 1, 0],
       [1, 0, 0, ..., 1, 0, 0],
       [1, 0, 0, ..., 1, 1, 1],
       ...,
       [0, 0, 1, ..., 0, 1, 0],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 0, 0, ..., 1, 1, 0]], shape=(7043, 7), dtype=object)

In [158]:
customer_churn.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [159]:
customer_churn.dropna(inplace=True,axis=0)

In [160]:
customer_churn.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,0,0,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,1,0,0,0,34,1,No,DSL,Yes,No,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,1,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,1,0,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,0,0,0,0,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


In [161]:
customer_churn.drop(columns=['customerID'],inplace=True)
customer_churn.head(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,1,0,0,0,34,1,No,DSL,Yes,No,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,1,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,1,0,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,0,0,0,0,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


In [162]:
# Identify categorical variables for one-hot encoding
categorical_columns = customer_churn.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", categorical_columns)
print(f"Booleancolumns {boolean_columns}")
print(f"\nNumber of categorical columns: {len(categorical_columns)}")

# Check unique values in each categorical column
for col in categorical_columns:
    print(f"\n{col}: {customer_churn[col].nunique()} unique values")
    print(customer_churn[col].unique())

Categorical columns: ['SeniorCitizen', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
Booleancolumns ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']

Number of categorical columns: 11

SeniorCitizen: 2 unique values
[0 1]

MultipleLines: 3 unique values
['No phone service' 'No' 'Yes']

InternetService: 3 unique values
['DSL' 'Fiber optic' 'No']

OnlineSecurity: 3 unique values
['No' 'Yes' 'No internet service']

OnlineBackup: 3 unique values
['Yes' 'No' 'No internet service']

DeviceProtection: 3 unique values
['No' 'Yes' 'No internet service']

TechSupport: 3 unique values
['No' 'Yes' 'No internet service']

StreamingTV: 3 unique values
['No' 'Yes' 'No internet service']

StreamingMovies: 3 unique values
['No' 'Yes' 'No internet service']

Contract: 3 unique values
['Month-to-month' 'One year' 'Two year']

PaymentMet

In [163]:
# Check the current dtypes of boolean columns
print("Boolean columns dtypes:")
for col in boolean_columns:
    print(f"{col}: {customer_churn[col].dtype}")
    print(f"  Unique values: {customer_churn[col].unique()}")

Boolean columns dtypes:
gender: int64
  Unique values: [0 1]
SeniorCitizen: object
  Unique values: [0 1]
Partner: int64
  Unique values: [1 0]
Dependents: int64
  Unique values: [0 1]
PhoneService: int64
  Unique values: [0 1]
PaperlessBilling: int64
  Unique values: [1 0]
Churn: int64
  Unique values: [0 1]


In [164]:
# Apply one-hot encoding with numeric dtype
customer_churn_encoded = pd.get_dummies(customer_churn, columns=categorical_columns, drop_first=True, dtype=int)
customer_churn_encoded.head()

,gender,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,SeniorCitizen_1,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,0,1,0,1,29.85,29.85,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
1,1,0,0,34,1,0,56.95,1889.50,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1
2,1,0,0,2,1,1,53.85,108.15,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
3,1,0,0,45,0,0,42.30,1840.75,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0
4,0,0,0,2,1,1,70.70,151.65,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [165]:
# Check the shape after encoding
print(f"Original shape: {customer_churn.shape}")
print(f"After one-hot encoding: {customer_churn_encoded.shape}")
print(f"\nNew columns created: {customer_churn_encoded.shape[1] - customer_churn.shape[1]}")

Original shape: (7032, 20)
After one-hot encoding: (7032, 31)

New columns created: 11


In [166]:
# Verify all columns are numeric
print("Data types after encoding:")
print(customer_churn_encoded.dtypes.value_counts())

Data types after encoding:
int64      29
float64     2
Name: count, dtype: int64


In [167]:
# normalize columns 'tenure', 'MonthlyCharges', 'TotalCharges'
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
columns_to_normalize = ['tenure', 'MonthlyCharges', 'TotalCharges']
customer_churn_encoded[columns_to_normalize] = scaler.fit_transform(customer_churn_encoded[columns_to_normalize])
customer_churn_encoded.head()

,gender,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,SeniorCitizen_1,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,0,0.000000,0,1,0.115423,0.001275,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0.464789,1,0,0.385075,0.215867,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1
2,1,0,0,0.014085,1,1,0.354229,0.010310,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
3,1,0,0,0.619718,0,0,0.239303,0.210241,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0
4,0,0,0,0.014085,1,1,0.521891,0.015330,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [168]:
# Create a complete preprocessing pipeline with custom transformer
import pickle 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

# Custom transformer for all the preprocessing steps
class CustomPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.boolean_columns = None
        
    def fit(self, X, y=None):
        # Identify boolean columns (columns with exactly 2 unique values)
        self.boolean_columns = [col for col in X.columns if X[col].nunique() == 2]
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # 1. Convert TotalCharges to numeric
        if 'TotalCharges' in X.columns:
            X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
        
        # 2. Convert SeniorCitizen to object type
        if 'SeniorCitizen' in X.columns:
            X['SeniorCitizen'] = X['SeniorCitizen'].astype('object')
        
        # 3. Map boolean columns (Yes/No, Female/Male) to 0/1
        for col in self.boolean_columns:
            if col in X.columns:
                unique_vals = set(X[col].dropna().unique())
                if unique_vals.issubset({'Yes', 'No'}):
                    X[col] = X[col].map({'Yes': 1, 'No': 0})
                elif unique_vals.issubset({'Female', 'Male'}):
                    X[col] = X[col].map({'Female': 0, 'Male': 1})
        
        # 4. DON'T drop missing values here - handle it separately to keep X and y aligned
        # X = X.dropna()  # REMOVED
        
        # 5. Drop customerID if exists
        if 'customerID' in X.columns:
            X = X.drop(columns=['customerID'])
        
        return X

# Define feature groups
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
# Categorical features will be determined after custom preprocessing

# Create the pipeline
preprocessing_pipeline = Pipeline([
    ('custom_preprocessing', CustomPreprocessor())
])

print("✅ Custom preprocessing pipeline created!")
print("\nThis pipeline will handle:")
print("  1. Convert TotalCharges to numeric")
print("  2. Convert SeniorCitizen to object")
print("  3. Map Yes/No and Female/Male to 0/1")
print("  4. Drop customerID column")
print("\nNote: NaN handling will be done before splitting to keep X and y aligned")

✅ Custom preprocessing pipeline created!

This pipeline will handle:
  1. Convert TotalCharges to numeric
  2. Convert SeniorCitizen to object
  3. Map Yes/No and Female/Male to 0/1
  4. Drop customerID column

Note: NaN handling will be done before splitting to keep X and y aligned


In [169]:
# Now create the complete pipeline with encoding and scaling
# Read fresh data to test the pipeline
customer_churn_test = pd.read_csv("../data/Telco_Customer_Churn.csv")

# Get categorical columns after custom preprocessing
temp_preprocessor = CustomPreprocessor()
temp_data = temp_preprocessor.fit_transform(customer_churn_test)
categorical_features = temp_data.select_dtypes(include=['object']).columns.tolist()

print(f"Categorical features for one-hot encoding: {categorical_features}")
print(f"Numerical features for scaling: {numerical_features}")

Categorical features for one-hot encoding: ['SeniorCitizen', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
Numerical features for scaling: ['tenure', 'MonthlyCharges', 'TotalCharges']


In [170]:
# Complete pipeline with all transformations
complete_pipeline = Pipeline([
    ('custom_preprocessing', CustomPreprocessor()),
    ('column_transformer', ColumnTransformer(
        transformers=[
            ('num', MinMaxScaler(), numerical_features),
            ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
        ],
        remainder='passthrough'  # Keep other columns (already encoded boolean columns)
    ))
])

print("✅ Complete pipeline created!")
print("\nPipeline steps:")
print("  1. Custom preprocessing (TotalCharges, boolean mapping, etc.)")
print("  2. MinMax scaling for numerical features")
print("  3. One-hot encoding for categorical features")
print("  4. Keep binary encoded features as-is")

✅ Complete pipeline created!

Pipeline steps:
  1. Custom preprocessing (TotalCharges, boolean mapping, etc.)
  2. MinMax scaling for numerical features
  3. One-hot encoding for categorical features
  4. Keep binary encoded features as-is


In [171]:
# Test the pipeline on fresh data
customer_churn_pipeline_test = pd.read_csv("../data/Telco_Customer_Churn.csv")

# Separate features and target
y = customer_churn_pipeline_test['Churn']
X = customer_churn_pipeline_test.drop(columns=['Churn'])

# Fit and transform
X_transformed = complete_pipeline.fit_transform(X)

print(f"Original shape: {X.shape}")
print(f"Transformed shape: {X_transformed.shape}")
print(f"✅ Pipeline successfully processed the data!")
print(f"\nFirst few rows of transformed data:")
print(X_transformed[:3])

Original shape: (7043, 20)
Transformed shape: (7043, 30)
✅ Pipeline successfully processed the data!

First few rows of transformed data:
[[0.01388889 0.11542289 0.0012751  0.         1.         0.
  0.         0.         0.         0.         0.         1.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         1.
  0.         0.         1.         0.         0.         1.        ]
 [0.47222222 0.38507463 0.21586661 0.         0.         0.
  0.         0.         0.         1.         0.         0.
  0.         1.         0.         0.         0.         0.
  0.         0.         1.         0.         0.         0.
  1.         1.         0.         0.         1.         0.        ]
 [0.02777778 0.35422886 0.01031041 0.         0.         0.
  0.         0.         0.         1.         0.         1.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  1.

In [172]:
# Save the pipeline for later use
with open('../preprocessor_pipeline.pkl', 'wb') as f:
    pickle.dump(complete_pipeline, f)

print("✅ Pipeline saved as 'preprocessor_pipeline.pkl'")
print("\nYou can now use this pipeline to transform new data:")
print("  1. Load: pipeline = pickle.load(open('preprocessor_pipeline.pkl', 'rb'))")
print("  2. Transform: X_new_transformed = pipeline.transform(X_new)")

✅ Pipeline saved as 'preprocessor_pipeline.pkl'

You can now use this pipeline to transform new data:
  1. Load: pipeline = pickle.load(open('preprocessor_pipeline.pkl', 'rb'))
  2. Transform: X_new_transformed = pipeline.transform(X_new)


In [173]:
#saving data to sqlite database 
conn = sqlite3.connect('../data/customer_churn.db')
customer_churn_dataframe = pd.read_csv("../data/Telco_Customer_Churn.csv")
customer_churn_dataframe.to_sql('customer_churn', conn, if_exists='replace', index=False)
conn.close()

In [174]:
from sqlalchemy import create_engine 
from sqlalchemy import text

engine = create_engine('sqlite:///../data/customer_churn.db')
with engine.connect() as connection:
    result = connection.execute(text("SELECT * FROM customer_churn LIMIT 5"))
    rows = result.fetchall()
    for row in rows:
        print(row)

('7590-VHVEG', 'Female', 0, 'Yes', 'No', 1, 'No', 'No phone service', 'DSL', 'No', 'Yes', 'No', 'No', 'No', 'No', 'Month-to-month', 'Yes', 'Electronic check', 29.85, '29.85', 'No')
('5575-GNVDE', 'Male', 0, 'No', 'No', 34, 'Yes', 'No', 'DSL', 'Yes', 'No', 'Yes', 'No', 'No', 'No', 'One year', 'No', 'Mailed check', 56.95, '1889.5', 'No')
('3668-QPYBK', 'Male', 0, 'No', 'No', 2, 'Yes', 'No', 'DSL', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'Month-to-month', 'Yes', 'Mailed check', 53.85, '108.15', 'Yes')
('7795-CFOCW', 'Male', 0, 'No', 'No', 45, 'No', 'No phone service', 'DSL', 'Yes', 'No', 'Yes', 'Yes', 'No', 'No', 'One year', 'No', 'Bank transfer (automatic)', 42.3, '1840.75', 'No')
('9237-HQITU', 'Female', 0, 'No', 'No', 2, 'Yes', 'No', 'Fiber optic', 'No', 'No', 'No', 'No', 'No', 'No', 'Month-to-month', 'Yes', 'Electronic check', 70.7, '151.65', 'Yes')


In [175]:
#return data from sqlite database as a dataframe the split the data then preprocess train and test data seperatley 

def load_data_from_db(db_path,table_name):
    engine =create_engine(f"sqlite:///{db_path}")
    with engine.connect() as connection:
        query = f"select * from {table_name}"
        df = pd.read_sql(query,connection)
    return df

In [176]:
customer_churn = load_data_from_db(db_path="../data/customer_churn.db",table_name="customer_churn")
customer_churn.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [177]:
# Drop NaN values BEFORE splitting to keep X and y aligned
customer_churn_clean = customer_churn.dropna()

# split data set into test and train 
from sklearn.model_selection import train_test_split

X = customer_churn_clean.drop(columns=['Churn'])
y = customer_churn_clean['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dataset shape after removing NaN: {customer_churn_clean.shape}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Dataset shape after removing NaN: (7043, 21)
X_train shape: (5634, 20)
X_test shape: (1409, 20)


In [178]:
X_train.head()
y_train

3738     No
3151     No
4860     No
3867     No
3810     No
       ... 
6303     No
6227    Yes
4673    Yes
2710     No
5639     No
Name: Churn, Length: 5634, dtype: object

In [179]:
# Fit the pipeline on training data and transform both train and test
X_train_transformed = complete_pipeline.fit_transform(X_train)
X_test_transformed = complete_pipeline.transform(X_test)  # Only transform, don't fit!

print(f"X_train original shape: {X_train.shape}")
print(f"X_train transformed shape: {X_train_transformed.shape}")
print(f"\nX_test original shape: {X_test.shape}")
print(f"X_test transformed shape: {X_test_transformed.shape}")
print(f"\n✅ Data successfully transformed!")

X_train original shape: (5634, 20)
X_train transformed shape: (5634, 30)

X_test original shape: (1409, 20)
X_test transformed shape: (1409, 30)

✅ Data successfully transformed!


In [180]:
# Also transform the target variable (Churn: Yes/No to 1/0)
y_train_encoded = y_train.map({'Yes': 1, 'No': 0})
y_test_encoded = y_test.map({'Yes': 1, 'No': 0})

print(f"y_train shape: {y_train_encoded.shape}")
print(f"y_test shape: {y_test_encoded.shape}")
print(f"\nClass distribution in training set:")
print(y_train_encoded.value_counts(normalize=True))

y_train shape: (5634,)
y_test shape: (1409,)

Class distribution in training set:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64


In [189]:
#now training the model using XGBOOST and Optuna for hyperparameter tuning
import xgboost as xgb
import optuna
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def objective(trial):
    param = {
        'verbosity': 0,  # Reduce output noise
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'use_label_encoder': False,
        
        # Tree structure parameters - EXPANDED RANGE
        'max_depth': trial.suggest_int('max_depth', 4, 15),  # Deeper trees
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),  # Wider range
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),  # Helps with imbalanced data
        
        # Learning parameters - MORE ESTIMATORS
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),  # Log scale
        'n_estimators': trial.suggest_int('n_estimators', 200, 2000),  # More trees
        
        # Sampling parameters - EXPANDED
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.6, 1.0),  # NEW
        
        # Regularization parameters - EXPANDED (FIXED: log=True requires low > 0)
        'reg_alpha': trial.suggest_float('reg_alpha', 0.001, 10.0, log=True),  # L1 - FIXED
        'reg_lambda': trial.suggest_float('reg_lambda', 0.001, 10.0, log=True),  # L2 - FIXED
        
        # Handle imbalanced data - CRITICAL FOR ACCURACY
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 10.0),
        
        # Additional parameters
        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        'max_leaves': trial.suggest_int('max_leaves', 0, 256),  # For lossguide
        'max_bin': trial.suggest_int('max_bin', 128, 512),  # Histogram size
        
        'random_state': 42,
        'tree_method': 'hist',  # Faster training
        'enable_categorical': False
    }
    
    model = xgb.XGBClassifier(**param)
    model.fit(X_train_transformed, y_train_encoded, 
              eval_set=[(X_test_transformed, y_test_encoded)],
              verbose=False)
    
    # Get probabilities for AUC (better than binary predictions)
    y_pred_proba = model.predict_proba(X_test_transformed)[:, 1]
    y_pred = model.predict(X_test_transformed)
    
    # Calculate multiple metrics for comparison
    auc = roc_auc_score(y_test_encoded, y_pred_proba)
    accuracy = accuracy_score(y_test_encoded, y_pred)
    precision = precision_score(y_test_encoded, y_pred)
    recall = recall_score(y_test_encoded, y_pred)
    f1 = f1_score(y_test_encoded, y_pred)
    
    # Store additional metrics as trial attributes
    trial.set_user_attr('accuracy', accuracy)
    trial.set_user_attr('precision', precision)
    trial.set_user_attr('recall', recall)
    trial.set_user_attr('f1', f1)
    
    # Optimize for accuracy instead of AUC if you want higher accuracy
    # return accuracy  # Use this if you want to optimize for accuracy
    return auc  # Or keep AUC for better overall ranking

In [190]:
# Run the optimization with MORE TRIALS
study = optuna.create_study(direction='maximize', study_name='xgboost_churn')
study.optimize(objective, n_trials=100, show_progress_bar=True)  # Increased from 50 to 100

print("=" * 50)
print("OPTIMIZATION RESULTS")
print("=" * 50)
print(f"\nBest AUC: {study.best_value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

print(f"\nBest trial metrics:")
print(f"  Accuracy:  {study.best_trial.user_attrs['accuracy']:.4f}")
print(f"  Precision: {study.best_trial.user_attrs['precision']:.4f}")
print(f"  Recall:    {study.best_trial.user_attrs['recall']:.4f}")
print(f"  F1 Score:  {study.best_trial.user_attrs['f1']:.4f}")
print(f"  AUC:       {study.best_value:.4f}")

# Show top 5 trials by accuracy
print("\n" + "=" * 50)
print("TOP 5 TRIALS BY ACCURACY")
print("=" * 50)
trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values('user_attrs_accuracy', ascending=False).head(5)
print(trials_df[['number', 'user_attrs_accuracy', 'user_attrs_precision', 'user_attrs_recall', 'user_attrs_f1', 'value']])

[I 2025-11-24 22:37:24,729] A new study created in memory with name: xgboost_churn
Best trial: 0. Best value: 0.842692:   1%|          | 1/100 [00:00<01:35,  1.04it/s]

[I 2025-11-24 22:37:25,693] Trial 0 finished with value: 0.8426916221033869 and parameters: {'max_depth': 9, 'min_child_weight': 8, 'gamma': 1.325753381652478, 'max_delta_step': 3, 'learning_rate': 0.024470774762296005, 'n_estimators': 322, 'subsample': 0.931159613876554, 'colsample_bytree': 0.8301313504709691, 'colsample_bylevel': 0.8117411708725282, 'colsample_bynode': 0.8903285778183699, 'reg_alpha': 0.0504718628669293, 'reg_lambda': 7.550020850004477, 'scale_pos_weight': 7.075069376750838, 'grow_policy': 'depthwise', 'max_leaves': 244, 'max_bin': 300}. Best is trial 0 with value: 0.8426916221033869.


Best trial: 1. Best value: 0.843168:   2%|▏         | 2/100 [00:01<01:13,  1.33it/s]

[I 2025-11-24 22:37:26,295] Trial 1 finished with value: 0.8431682554444704 and parameters: {'max_depth': 8, 'min_child_weight': 1, 'gamma': 3.2603367682408653, 'max_delta_step': 9, 'learning_rate': 0.04265071676236977, 'n_estimators': 475, 'subsample': 0.9129252071722052, 'colsample_bytree': 0.9895444051252255, 'colsample_bylevel': 0.9788325688322477, 'colsample_bynode': 0.6004306717652481, 'reg_alpha': 0.006608779868688582, 'reg_lambda': 0.027592438878234815, 'scale_pos_weight': 3.489388107376015, 'grow_policy': 'depthwise', 'max_leaves': 104, 'max_bin': 443}. Best is trial 1 with value: 0.8431682554444704.


Best trial: 1. Best value: 0.843168:   3%|▎         | 3/100 [00:03<02:11,  1.36s/it]

...

[I 2025-11-24 22:44:56,521] Trial 97 finished with value: 0.8504611330698286 and parameters: {'max_depth': 9, 'min_child_weight': 9, 'gamma': 0.8929054248544588, 'max_delta_step': 9, 'learning_rate': 0.004330782904065715, 'n_estimators': 1137, 'subsample': 0.6866589654365732, 'colsample_bytree': 0.6634596941678506, 'colsample_bylevel': 0.6822597714438147, 'colsample_bynode': 0.6458048729679129, 'reg_alpha': 4.946612288460817, 'reg_lambda': 0.012309193334438097, 'scale_pos_weight': 1.3398252063032565, 'grow_policy': 'lossguide', 'max_leaves': 115, 'max_bin': 496}. Best is trial 95 with value: 0.8510126843886434.


Best trial: 95. Best value: 0.851013:  99%|█████████▉| 99/100 [07:40<00:07,  7.54s/it]

[I 2025-11-24 22:45:05,645] Trial 98 finished with value: 0.8498656643157921 and parameters: {'max_depth': 9, 'min_child_weight': 9, 'gamma': 0.9548645930057094, 'max_delta_step': 10, 'learning_rate': 0.004067026686433043, 'n_estimators': 1130, 'subsample': 0.6883110694194516, 'colsample_bytree': 0.6690432469253862, 'colsample_bylevel': 0.7097993952056927, 'colsample_bynode': 0.6471313712692118, 'reg_alpha': 1.9130557046552674, 'reg_lambda': 0.01145006533722342, 'scale_pos_weight': 1.3754360527168326, 'grow_policy': 'lossguide', 'max_leaves': 39, 'max_bin': 510}. Best is trial 95 with value: 0.8510126843886434.


Best trial: 95. Best value: 0.851013: 100%|██████████| 100/100 [07:53<00:00,  4.73s/it]

[I 2025-11-24 22:45:17,809] Trial 99 finished with value: 0.8491694437985997 and parameters: {'max_depth': 9, 'min_child_weight': 10, 'gamma': 0.8142909751372156, 'max_delta_step': 9, 'learning_rate': 0.0031043642395081382, 'n_estimators': 1312, 'subsample': 0.6507757087521114, 'colsample_bytree': 0.6206994191396439, 'colsample_bylevel': 0.6607216641650279, 'colsample_bynode': 0.7008615671670254, 'reg_alpha': 0.06020981206808512, 'reg_lambda': 0.005252129274238226, 'scale_pos_weight': 1.9262666951007783, 'grow_policy': 'lossguide', 'max_leaves': 76, 'max_bin': 496}. Best is trial 95 with value: 0.8510126843886434.
OPTIMIZATION RESULTS

Best AUC: 0.8510

Best hyperparameters:
  max_depth: 9
  min_child_weight: 8
  gamma: 0.7537914553607383
  max_delta_step: 10
  learning_rate: 0.00520964879173031
  n_estimators: 1280
  subsample: 0.6862913812669756
  colsample_bytree: 0.6638660546347694
  colsample_bylevel: 0.7336502387307278
  colsample_bynode: 0.6989921711792295
  reg_alpha: 5.0277091

In [191]:
# Train final model with best parameters
best_model = xgb.XGBClassifier(**study.best_params, random_state=42)
best_model.fit(X_train_transformed, y_train_encoded)

# Make predictions
y_pred_proba = best_model.predict_proba(X_test_transformed)[:, 1]
y_pred = best_model.predict(X_test_transformed)

print("\n" + "=" * 50)
print("FINAL MODEL EVALUATION")
print("=" * 50)
print(f"\nAccuracy:  {accuracy_score(y_test_encoded, y_pred):.4f}")
print(f"Precision: {precision_score(y_test_encoded, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test_encoded, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test_encoded, y_pred):.4f}")
print(f"AUC:       {roc_auc_score(y_test_encoded, y_pred_proba):.4f}")


FINAL MODEL EVALUATION

Accuracy:  0.7991
Precision: 0.6213
Recall:    0.6230
F1 Score:  0.6222
AUC:       0.8510


### Note on Accuracy vs AUC

**Important**: 72% accuracy might actually be good for churn prediction! Here's why:

1. **Class imbalance**: If 73% of customers don't churn, a model predicting "No churn" for everyone gets 73% accuracy but is useless
2. **Business value**: It's better to correctly identify churners (high recall) than overall accuracy
3. **90% accuracy is unrealistic** for most real-world churn datasets

**If you still want to optimize for accuracy:**
- Change `return auc` to `return accuracy` in the objective function above
- But be aware this might hurt the model's ability to identify actual churners

**Better approach**: Focus on F1-score or recall for churners, not raw accuracy

In [192]:
# Save the best model
with open('../best_xgboost_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("\n✅ Model saved as 'best_xgboost_model.pkl'")


✅ Model saved as 'best_xgboost_model.pkl'


### Why AUC instead of Accuracy, Precision, or Recall?

**AUC (Area Under ROC Curve)** is often preferred for binary classification, especially with imbalanced datasets:

1. **Threshold-independent**: AUC measures performance across all classification thresholds, not just one
2. **Handles class imbalance**: Works well even when classes are imbalanced (churn is typically minority class)
3. **Balanced metric**: Considers both true positive rate and false positive rate
4. **Business value**: Shows model's ability to rank customers by churn probability

**When to use each metric:**
- **Accuracy**: When classes are balanced and false positives/negatives have equal cost
- **Precision**: When false positives are costly (e.g., spam detection - don't want to mark real emails as spam)
- **Recall**: When false negatives are costly (e.g., disease detection - don't want to miss sick patients)
- **AUC**: When you want overall ranking ability and threshold flexibility

For churn prediction, **AUC is ideal** because:
- We want to identify high-risk customers (ranking matters)
- Business can adjust threshold based on retention campaign capacity
- Class imbalance is common (more non-churners than churners)